# Random forest tuning

One grid search per analysis, scored with grouped CV folds over the **training countries only** — the 20 countries `models.ipynb` holds out are never seen here. The winning hyperparameters are saved to `data/hyperparameters/rf_<name>.json`, which `models.ipynb` loads:

- `rf_union_density.json`
- `rf_electoral_democracy.json`
- `rf_electoral_democracy_no_vdem.json`

In [1]:
import json
from datetime import date
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

In [2]:
# NOTE: this block must stay identical to the one in models.ipynb, so the
# search never sees the test countries that models.ipynb holds out
df = pd.read_csv("../data/merged.csv")

ID_COLS = ["iso3", "country", "Unnamed: 0"]
DROP = ["emp_services_pct",     # emp shares sum to exactly 100
        "services_va_pct_gdp",  # value-added shares nearly sum to 100
        "emp_pop_ratio",        # ≈ lfp_rate × (1 − unemployment)
        "self_employed_pct",    # r = 0.996 with vulnerable_emp_pct
        "exports_pct_gdp",      # r = 0.987 with trade_pct_gdp
        "liberal_democracy"]    # r = 0.981 with electoral_democracy
# V-Dem builds electoral_democracy (v2x_polyarchy) from freedom_of_association
# (v2x_frassoc_thick), which is itself built from cso_entry_exit and
# cso_repression; civil_society_index shares those two inputs (codebook v16).
# Predicting democracy from them would be predicting the index from its parts
DEM_INGREDIENTS = ["freedom_of_association", "cso_entry_exit",
                   "cso_repression", "civil_society_index"]
# every V-Dem column in merged.csv (VDEM_NAMES in cleaning.ipynb), for the
# democracy model with no V-Dem predictors at all
VDEM_COLS = ["freedom_of_association", "civil_society_index", "electoral_democracy",
             "liberal_democracy", "cso_repression", "cso_entry_exit",
             "egalitarian_component", "equal_resource_distribution",
             "power_by_socioeconomic_position", "state_mass_org_engagement",
             "regime_support_working_class", "govt_ideology_socialist"]
# predictors from the union-density literature -- Ghent system, state-run vs
# independent unions, labour's political power -- used by the union density
# model only
UNION_ONLY = ["ghent", "state_mass_org_engagement",
              "regime_support_working_class", "govt_ideology_socialist"]

cols_union = [c for c in df if c not in ID_COLS + ["union_density"] + DROP]
cols_dem = [c for c in df if c not in ID_COLS + ["electoral_democracy"] + DROP + DEM_INGREDIENTS + UNION_ONLY]
cols_dem_novdem = [c for c in df if c not in ID_COLS + VDEM_COLS + DROP + UNION_ONLY]

# rows must be complete in every column except the union-only additions, so
# adding them can't change the democracy analyses or the train/test split. the
# union model drops its own incomplete rows, if any
df = df.dropna(subset=[c for c in df.columns if c not in UNION_ONLY]).reset_index(drop=True)

random_state = 42

# one train/test split: models.ipynb fits its OLS summaries on the training
# rows, and tuning.ipynb tunes on the training countries only. whole countries
# are held out, so no country appears in both halves
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=random_state)
train_idx, test_idx = next(splitter.split(df, groups=df["iso3"]))
train, test = df.loc[train_idx], df.loc[test_idx]

PARAMS_DIR = Path("../data/hyperparameters")

PARAMS_DIR.mkdir(parents=True, exist_ok=True)

print(f"tuning on {len(train)} rows / {train['iso3'].nunique()} countries")
print(f"features -- union density: {len(cols_union)} | democracy: {len(cols_dem)} | "
      f"democracy, no V-Dem: {len(cols_dem_novdem)}")

tuning on 787 rows / 80 countries
features -- union density: 39 | democracy: 31 | democracy, no V-Dem: 28


In [3]:
# folds group by country as well: a country is never in a fold's train and
# validation halves at once, which is how models.ipynb will be judged
GRID = {
    "max_features": [0.2, 0.33, 0.5, 1.0],
    "min_samples_leaf": [1, 2, 5, 10],
    "max_samples": [None, 0.7],
}
N_ESTIMATORS = 1000


def tune(target, feature_cols, name=None):
    # `name` keys the saved file, so two analyses of one target don't collide
    name = name or target
    # each analysis uses the training rows complete for its own features
    rows = train.dropna(subset=feature_cols)
    X = rows[feature_cols]
    y = rows[target]
    groups = rows["iso3"]

    scaler = StandardScaler().fit(X)
    X_s = pd.DataFrame(scaler.transform(X), columns=feature_cols, index=X.index)

    search = GridSearchCV(
        RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=random_state, n_jobs=-1),
        GRID,
        cv=GroupKFold(5),
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    ).fit(X_s, y, groups=groups)

    results = pd.DataFrame(search.cv_results_)
    results["rmse"] = -results["mean_test_score"]
    best = results.loc[results["rmse"].idxmin()]

    print(f"{name}: {len(results)} combinations tried\n")
    print(results.nsmallest(10, "rmse")[
        ["param_max_features", "param_min_samples_leaf", "param_max_samples", "rmse", "std_test_score"]
    ].to_string(index=False))

    # if the spread across all combinations is small next to the fold-to-fold
    # noise, trust the direction the winner points, not its exact values
    print(f"\nRMSE across all combinations: {results.rmse.min():.3f} to {results.rmse.max():.3f}"
          f"  (a spread of {results.rmse.max() - results.rmse.min():.3f})")
    print(f"fold-to-fold spread within the best combination: {best['std_test_score']:.3f}")

    export = {
        "name": name,
        "target": target,
        "features": list(feature_cols),
        "params": {"n_estimators": N_ESTIMATORS, **search.best_params_},
        "cv_rmse": round(float(-search.best_score_), 4),
        "cv_rmse_sd": round(float(best["std_test_score"]), 4),
        "searched": {k: [str(v) for v in vals] for k, vals in GRID.items()},
        "n_combinations": int(len(results)),
        "train_rows": int(len(rows)),
        "train_countries": int(groups.nunique()),
        "random_state": random_state,
        "created": date.today().isoformat(),
    }

    path = PARAMS_DIR / f"rf_{name}.json"
    with open(path, "w") as f:
        json.dump(export, f, indent=2)

    print(f"\nsaved to {path}: {export['params']}")
    return results

## Union density

In [4]:
results_union = tune("union_density", cols_union)

union_density: 32 combinations tried

 param_max_features  param_min_samples_leaf param_max_samples      rmse  std_test_score
               0.33                       1              None 12.720471        3.560187
               0.33                       5              None 12.785193        3.385051
               0.33                       2              None 12.785274        3.490811
               0.50                       2               0.7 12.799896        3.270982
               0.50                       1              None 12.815553        3.441487
               0.20                       2               0.7 12.816727        3.560165
               0.20                       2              None 12.818382        3.657973
               0.20                       1               0.7 12.828519        3.750404
               0.50                       1               0.7 12.828667        3.273040
               0.33                      10              None 12.829408        3.2

## Electoral democracy

The four `DEM_INGREDIENTS` features are excluded, and `union_density` is one of the predictors.

In [5]:
results_dem = tune("electoral_democracy", cols_dem)

electoral_democracy: 32 combinations tried

 param_max_features  param_min_samples_leaf param_max_samples     rmse  std_test_score
               1.00                       5              None 0.137912        0.031480
               1.00                       2              None 0.138227        0.027640
               1.00                       5               0.7 0.138239        0.032676
               1.00                       1              None 0.138273        0.026349
               1.00                      10              None 0.138598        0.034277
               1.00                       2               0.7 0.138642        0.029935
               1.00                       1               0.7 0.138851        0.028929
               0.33                       2               0.7 0.139475        0.030868
               0.50                       1              None 0.139654        0.027837
               0.33                       1              None 0.139790        0.029741

## Electoral democracy, no V-Dem predictors

Same target, but every V-Dem column (`VDEM_COLS`) is removed from the predictors — World Bank variables and `union_density` only. Saved under its own name so it doesn't overwrite the section above.

In [6]:
results_novdem = tune("electoral_democracy", cols_dem_novdem, name="electoral_democracy_no_vdem")

electoral_democracy_no_vdem: 32 combinations tried

 param_max_features  param_min_samples_leaf param_max_samples     rmse  std_test_score
               0.20                      10               0.7 0.166520        0.033791
               0.20                      10              None 0.167854        0.033655
               0.20                       2               0.7 0.167943        0.031739
               0.20                       5               0.7 0.168118        0.032655
               0.20                       1               0.7 0.168904        0.030576
               0.20                       5              None 0.168948        0.032311
               0.20                       2              None 0.169143        0.029833
               0.33                      10               0.7 0.169922        0.034335
               0.20                       1              None 0.169947        0.029797
               0.33                      10              None 0.171153        